In [ ]:
%%bash
# Remove any R3 controls from an export
jq -c --slurpfile pasted "r2.json" '
  ($pasted[0]
    | map(select(.element_type == "security_requirement"))
    | map(.element_identifier | sub("^SR-"; ""))
  ) as $allowedSecurityRequirementIds

  | (
      $allowedSecurityRequirementIds
      | map(sub("\\.[a-z]$"; ""))
      | unique
    ) as $allowedBaseRequirementIds

  | (
      (.securityRequirements // [])
      | map(select(.id as $id | $allowedSecurityRequirementIds | index($id)))
    ) as $filteredSecurityRequirements

  | (
      (.evidenceRequirements // [])
      | map(select(.requirement_id as $rid | $allowedBaseRequirementIds | index($rid)))
    ) as $filteredEvidenceRequirements

  | (
      $filteredEvidenceRequirements
      | map(.evidence_id)
      | unique
    ) as $keptEvidenceIds

  | {
      version: 4,
      securityRequirements: $filteredSecurityRequirements,
      evidenceRequirements: $filteredEvidenceRequirements,
      evidence: (
        (.evidence // [])
        | map(select(.id as $id | $keptEvidenceIds | index($id)))
      )
    }
' cmmc-800-171-rev-2-export-*.json > cmmc-800-171-rev-2-only-export.json